# Module 12 — LangChain Tool-Calling + Pricing AI Agent

This notebook evaluates the PriceMind AI Agent pipeline end-to-end.

**Architecture:**
```
User → FastAPI /agent/query → PricingAgent → Tool Selection → Service Layer → Tool Result → Stub/LLM Synthesis → Grounded Response
```


In [ ]:
import sys
from pathlib import Path

# Add project root and backend to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))
print('Project root:', ROOT)


## 1. Safety Guardrails


In [ ]:
from agent.safety import (
    contains_forbidden_action,
    validate_price,
    flag_high_risk_change,
)

# Test forbidden action detection
print('Forbidden action tests:')
print('  apply price:', contains_forbidden_action('Please apply the price change'))  # True
print('  safe query:', contains_forbidden_action('What is the elasticity?'))  # False

# Test high-risk flag
print('\nHigh risk change (>30%):')
print(' ', flag_high_risk_change(100.0, 145.0))
print('Safe change (<30%):')
print(' ', flag_high_risk_change(100.0, 115.0))  # None


## 2. Intent Router


In [ ]:
from agent.agent import _classify_intent, _extract_product_id, _extract_price

test_queries = [
    'What is the margin floor policy?',
    'Tell me about SKU-8921-PRO',
    'Is SKU-8921-PRO elastic or inelastic?',
    'What is the demand forecast for next 2 weeks?',
    'What if we price at $430?',
    'What is the optimal price for SKU-8921-PRO?',
    'Why was the price recommendation made?',
]

print(f'{'Query':<50} → Tools')
print('-' * 80)
for q in test_queries:
    tools = _classify_intent(q)
    print(f'{q[:50]:<50} → {tools}')

print('\nEntity extraction:')
print('  SKU from text:', _extract_product_id('Tell me about SKU-8921-PRO'))
print('  Price from text:', _extract_price('What if we price at $430.50?'))


## 3. Agent Query — Stub Path (No LLM Key Required)


In [ ]:
from agent.agent import PricingAgent
from agent.config import AgentConfig

# Create agent in stub mode
cfg = AgentConfig(llm_provider='stub')
agent = PricingAgent(config=cfg)

print('Agent LLM backend:', 'stub (no API key)' if agent._llm_chain is None else 'LLM connected')
print('Tools registered:', len(agent._llm_chain.tools if agent._llm_chain else []), 'N/A in stub mode')


In [ ]:
# Test forbidden action blocking
result = agent.query('Please apply the price change now')
print('Forbidden action response:')
print(result['answer'])
print('Tools used:', result['tools_used'])


In [ ]:
# Test policy question → RAG
from unittest.mock import patch, MagicMock

with patch('agent.tools.rag_tools.rag_service') as mock_rag:
    mock_rag.query.return_value = MagicMock(
        answer='The corporate margin floor is 20% gross margin.',
        is_grounded=True,
        confidence_score=0.85,
        sources=[{'title': 'Pricing Policy', 'category': 'pricing', 'chunk_preview': 'margin floor = 20%'}],
    )
    result = agent.query('What is the margin floor policy?')

print('Tools used:', result['tools_used'])
print('Answer:')
print(result['answer'])


## 4. Tool Registry


In [ ]:
from agent.tools import ALL_TOOLS

print(f'Total tools registered: {len(ALL_TOOLS)}')
print()
for tool in ALL_TOOLS:
    desc_first_line = tool.description.strip().split('\n')[0][:70]
    print(f'  • {tool.name:<42} → {desc_first_line}')


## 5. FastAPI Endpoint Integration


In [ ]:
from fastapi.testclient import TestClient
from fastapi import FastAPI
from app.api.v1.endpoints.agent import router
from unittest.mock import patch, MagicMock

app = FastAPI()
app.include_router(router, prefix='/agent')
client = TestClient(app)

# Test the endpoint with a mocked agent
with patch('app.api.v1.endpoints.agent.get_agent') as mock_get:
    mock_agent = MagicMock()
    mock_agent.query.return_value = {
        'answer': 'The corporate margin floor is 20%.',
        'tools_used': ['search_knowledge_base_tool'],
        'sources': [{'title': 'Pricing Policy', 'category': 'pricing', 'chunk_preview': '...', 'similarity_score': 0.85}],
        'data': {},
    }
    mock_get.return_value = mock_agent
    
    resp = client.post('/agent/query', json={'message': 'What is the margin floor policy?'})

print('Status:', resp.status_code)
print('Response keys:', list(resp.json().keys()))
print('Answer:', resp.json()['answer'])
print('Tools used:', resp.json()['tools_used'])
print('Sources:', len(resp.json()['sources']))


## 6. Summary


In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_agent.py', '-v', '--tb=short', '-q'],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
